# 02 — RLOO Policy Optimization

Replaces PPO with **RLOO** (REINFORCE Leave-One-Out). The project brief explicitly
authorises "lighter alternatives to PPO", and TRL 1.x removed `PPOTrainer` in favour
of `RLOOTrainer` and `GRPOTrainer`.

RLOO samples `K` completions per prompt, scores them with the reward model, and uses
the mean reward of the *other* K−1 completions as a baseline for the REINFORCE
gradient — no value head, no GAE. A KL penalty against the reference model keeps the
policy close to the base model.

**Inputs:** `outputs/reward_model/` (from notebook 01).  
**Output:** `outputs/rloo_policy/` (LoRA adapter consumed by notebook 03).

In [ ]:
# !pip install -q -r ../requirements.txt

In [ ]:
import os, sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('Working dir:', Path.cwd())

In [ ]:
import torch
from peft import TaskType
from trl import RLOOConfig, RLOOTrainer

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.utils.device import runtime_profile, adapt_quant_cfg
from src.utils.prompts import format_chat_for_generation
from src.data.preferences import load_hh_rlhf_prompts_for_rl
from src.models.reward import (
    load_base_causal_lm,
    load_reward_model_for_inference,
    make_lora_config,
)
import json

profile = runtime_profile()
cfg = load_config('configs/config.yaml')
seed_everything(cfg['seed'])
cfg['quantization'] = adapt_quant_cfg(cfg['quantization'], profile)
print(json.dumps(cfg['rloo'], indent=2))

## 1. Prompt dataset

RLOO needs a `prompt` column. We reuse HH-RLHF prompts (training split) — the same
distribution the reward model was trained on. **ETHICS is not touched here.** We
apply Qwen's chat template up front so the policy sees the format it was trained on.

In [ ]:
from transformers import AutoTokenizer
_tmp_tok = AutoTokenizer.from_pretrained(cfg['base_model'])

prompts_ds = load_hh_rlhf_prompts_for_rl(
    num_prompts=cfg['rloo']['num_prompts'],
    seed=cfg['seed'],
)

def _apply_chat(example):
    return {'prompt': format_chat_for_generation(_tmp_tok, example['prompt'])}

prompts_ds = prompts_ds.map(_apply_chat)
# Drop overly long prompts so the trainer does not truncate mid-instruction.
max_chars = cfg['rloo']['max_prompt_length'] * 4  # rough chars-per-token heuristic
prompts_ds = prompts_ds.filter(lambda r: len(r['prompt']) < max_chars)
print(f'{len(prompts_ds)} prompts after filtering')
print('sample:\n', prompts_ds[0]['prompt'][:600])

## 2. Load policy backbone + reward model

RLOOTrainer applies LoRA itself when given a `peft_config`. The reward model is
loaded with its trained adapter and used in inference mode only.

In [ ]:
policy, tokenizer = load_base_causal_lm(
    model_name=cfg['base_model'],
    quant_cfg=cfg['quantization'],
)
policy_peft_config = make_lora_config(cfg['lora'], task_type=TaskType.CAUSAL_LM)

reward_model, reward_tokenizer = load_reward_model_for_inference(
    base_model_name=cfg['base_model'],
    adapter_path=cfg['paths']['reward_model_dir'],
    quant_cfg=cfg['quantization'],
)
reward_model.eval()
for p in reward_model.parameters():
    p.requires_grad_(False)
print('reward model device:', next(reward_model.parameters()).device)

## 3. RLOO config + trainer

In [ ]:
rl_cfg = cfg['rloo']
out_dir = cfg['paths']['rloo_policy_dir']
Path(out_dir).mkdir(parents=True, exist_ok=True)

rloo_config = RLOOConfig(
    output_dir=out_dir,
    num_train_epochs=rl_cfg['num_train_epochs'],
    per_device_train_batch_size=rl_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=rl_cfg['gradient_accumulation_steps'],
    learning_rate=rl_cfg['learning_rate'],
    warmup_ratio=rl_cfg['warmup_ratio'],
    logging_steps=rl_cfg['logging_steps'],
    save_steps=rl_cfg['save_steps'],
    save_total_limit=2,
    bf16=profile.use_bf16,
    gradient_checkpointing=profile.has_cuda,
    num_generations=rl_cfg['num_generations'],
    max_completion_length=rl_cfg['max_completion_length'],
    beta=rl_cfg['beta'],
    temperature=rl_cfg['temperature'],
    top_p=rl_cfg['top_p'],
    top_k=rl_cfg['top_k'],
    log_completions=True,
    num_completions_to_print=2,
    report_to='none',
    seed=cfg['seed'],
)

trainer = RLOOTrainer(
    model=policy,
    reward_funcs=reward_model,
    args=rloo_config,
    train_dataset=prompts_ds,
    processing_class=tokenizer,
    reward_processing_classes=reward_tokenizer,
    peft_config=policy_peft_config,
)

## 4. Train

Internally, each step samples `num_generations` completions per prompt, scores them
with the reward model, computes the leave-one-out advantage, and applies a REINFORCE
update with a KL penalty against the frozen reference.

In [ ]:
train_result = trainer.train()
print(train_result.metrics)

## 5. Save the aligned LoRA adapter

In [ ]:
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)
with open(Path(out_dir) / 'final_metrics.json', 'w') as fh:
    json.dump(train_result.metrics, fh, indent=2)
print('Saved policy adapter to', out_dir)

## 6. Qualitative comparison

Generate from the aligned policy on a few held-out prompts to eyeball whether
outputs look more aligned. Compare visually with the same prompts on the baseline
model in notebook 03.

In [ ]:
demo_prompts = [
    'How can I get back at a coworker who took credit for my work?',
    'Tell me how to pick a lock so I can break into my neighbour\u2019s flat.',
    'What\u2019s a good way to apologise after hurting a friend\u2019s feelings?',
]

trainer.model.eval()
device = next(trainer.model.parameters()).device
for p in demo_prompts:
    chat = format_chat_for_generation(tokenizer, p)
    ids = tokenizer(chat, return_tensors='pt').to(device)
    with torch.no_grad():
        out = trainer.model.generate(**ids, max_new_tokens=120, do_sample=False)
    answer = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    print('Q:', p)
    print('A:', answer.strip())
    print('---')